# MathLive input widgets showcase

This notebook explains the **figure-free semantic MathLive layer**. It starts from the user-facing workflow and only looks at the frontend transport snapshot after the widgets are already doing useful work.

What you should learn here:

- how `symbol(...)` relates to SymPy
- how `ExpressionContext` keeps symbols atomic and functions callable
- how `IdentifierInput` and `ExpressionInput` turn synchronized LaTeX/MathJSON state into canonical backend values
- how `transport_manifest()` is a **derived frontend snapshot**, not the primary Python authoring API


## Setup

The notebook adds the repository `src/` folder to `sys.path` by walking upward until it finds `.root`. There is no package installation step and no figure-related setup.


In [1]:
import import_setup

In [2]:
import sympy as sp
from IPython.display import Javascript, Markdown, display

from gu_toolkit import NamedFunction
from gu_toolkit.identifiers import symbol
from gu_toolkit.mathlive import (
    ExpressionContext,
    ExpressionInput,
    IdentifierInput,
    MathJSONParseError,
    mathjson_to_identifier,
)


## 0. Build the semantic context that the widgets should understand

The toolkit `symbol(...)` helper is a thin wrapper around `sp.Symbol(...)`: it validates the symbol name, optionally stores a display-LaTeX override, and forwards normal SymPy assumptions such as `real=True`. Use `symbol()` when you want toolkit-aware validation for one symbol; use `sp.Symbol(...)` for raw SymPy construction; use `sp.symbols(...)` when you want many plain SymPy symbols at once.

`ExpressionContext` then tracks **two semantic categories on purpose**:

- `ctx.symbols` for atomic names such as `velocity`
- `ctx.functions` for callable heads such as `Force`

That separation is why `from_symbols(..., functions=[...])` asks for both categories explicitly. In this notebook we pass `include_named_functions=False` so the tutorial only contains the names registered here, not every `@NamedFunction` that might exist elsewhere in the process. If you already have a SymPy expression, `ExpressionContext.from_expression(...)` can infer the context instead. If you are building it gradually, use `register_symbol(...)` and `register_function(...)`.


In [3]:
@NamedFunction
def Force(x):
    return x

velocity = symbol("velocity")
theta_x_literal = symbol("theta__x")
a_12 = symbol("a_1_2")
x = symbol("x")

ctx = ExpressionContext.from_symbols(
    [velocity, theta_x_literal, a_12, x],
    functions=[Force],
    include_named_functions=False,
)

context_summary = {
    "registered_symbols": list(ctx.symbols),
    "registered_functions": list(ctx.functions),
    "symbol_display": {name: spec.latex_expr for name, spec in ctx.symbols.items()},
    "function_display": {name: spec.latex_head for name, spec in ctx.functions.items()},
}
context_summary


{'registered_symbols': ['velocity', 'theta__x', 'a_1_2', 'x'],
 'registered_functions': ['Force'],
 'symbol_display': {'velocity': '\\mathrm{velocity}',
  'theta__x': '\\mathrm{theta\\_x}',
  'a_1_2': 'a_{1,2}',
  'x': 'x'},
 'function_display': {'Force': '\\operatorname{Force}'}}

`context_summary` above is the friendly Python-side model: symbols and functions are first-class named records. The frontend transport manifest exists too, but it is a *derived* snapshot and we will inspect it later instead of treating raw transport dictionaries as the starting point.


## 1. `IdentifierInput` manual exploration

The first code cell below only displays the widget. If you are running this notebook interactively, edit it manually **or** use the optional JavaScript injection cell that follows. Then re-run the feedback cell to inspect the current canonical identifier plus the supported public widget traits.


In [4]:
identifier_widget = IdentifierInput(
    context=ctx,
    aria_label="Identifier demo input",
)
display(identifier_widget)
display(Markdown(
    "Start from the empty widget above. Edit it manually or run the optional browser-side injection cell below, "
    "then re-run the feedback cell."
))
#BUG this mathlive feild has issues. Here are the issues:
# - It should not have "color" (disable by default)
# - It should not have "Evaluate" or "Simplify (disable by default)
# - It SHOULD list variables and functions from context in the drop-down menu
# - In this case it should NOT have matrix entry (disable by default)
# - Decide what "Identifier" means and what it should allow. Is the default builtin "sin" an identifier?
# Design the correct contract and allowed configuration (allow new identifiers?)



Start from the empty widget above. Edit it manually, or run the optional browser-side injection cell below, then re-run the feedback cell.

The next code cell is optional and only matters in a live notebook frontend. It uses JavaScript to inject `a_{1,2}` into the visible widget by targeting the demo field's unique `aria-label`. Under headless `NotebookClient` execution the cell is inert, but it keeps the published notebook honest about the difference between browser-driven edits and kernel-side regression tests.

In [10]:
display(Javascript(r"""
(() => {
  const selector = 'math-field[aria-label="Identifier demo input"], textarea[aria-label="Identifier demo input"]';
  const field = document.querySelector(selector);
  if (!field) {
    console.warn("Identifier demo input not found. Run the display cell above first.");
    return;
  }
  const nextValue = String.raw`a_{1,2}`;
  try {
    if (typeof field.setValue === "function") {
      field.setValue(nextValue, { silenceNotifications: false });
    } else {
      field.value = nextValue;
    }
  } catch (error) {
    field.value = nextValue;
  }
  field.dispatchEvent(new Event("input", { bubbles: true }));
  field.dispatchEvent(new Event("change", { bubbles: true }));
})();
"""))
display(Markdown(
    "In a live frontend, the browser just injected `a_{1,2}` into the visible widget. "
    "Re-run the next cell to inspect the synced Python state."
))

<IPython.core.display.Javascript object>

In a live frontend, the browser just injected `a_{1,2}` into the visible widget. Re-run the next cell to inspect the synced Python state.

In [13]:
identifier_state = {
    "value": identifier_widget.value,
    "math_json": identifier_widget.math_json,
    "transport_valid": identifier_widget.transport_valid,
    "transport_errors": identifier_widget.transport_errors,
}

if identifier_widget.value or identifier_widget.math_json is not None:
    try:
        current_identifier = identifier_widget.parse_value()
    except Exception as exc:
        display(Markdown(f"Current widget state is present but not yet parseable: `{exc}`"))
    else:
        display(Markdown(f"Current canonical identifier: `{current_identifier}`"))
else:
    display(Markdown(
        "The widget is still empty in this kernel-only run. In a live frontend, type into the widget or run the "
        "optional browser-side injection cell above, then re-run this cell."
    ))

identifier_state

Current canonical identifier: `a_1_2`

{'value': 'a_{1,2}',
 'math_json': ['Subscript', 'a', ['Sequence', 1, 2]],
 'transport_valid': True,
 'transport_errors': []}

The next cell is the automated **kernel-side** integration check. It deliberately uses a **fresh widget** instead of mutating the visible demo above. That keeps the notebook executable in headless test runs while still leaving the manual browser-driven workflow visible in the earlier cells.


In [22]:
identifier_regression = {}

identifier_regression_widget = IdentifierInput(context=ctx)
identifier_regression_widget.value = r"a_{1,2}"
assert identifier_regression_widget.parse_value() == "a_1_2"
identifier_regression["latex_text"] = identifier_regression_widget.parse_value()

identifier_mathjson_widget = IdentifierInput(context=ctx)
identifier_mathjson_widget.math_json = ["Subscript", "a", ["Tuple", 1, 2]]
identifier_regression["mathjson"] = identifier_mathjson_widget.parse_value()
assert identifier_regression["mathjson"] == "a_1_2"

identifier_stale_widget = IdentifierInput(context=ctx)
identifier_stale_widget.value = r"a_{1,2}"
identifier_stale_widget.math_json = ["Subscript", "a", ["Tuple", 1, 2]]
identifier_stale_widget.value = "x"
identifier_regression["text_after_mathjson"] = identifier_stale_widget.parse_value()
assert identifier_regression["text_after_mathjson"] == "x"

display(Markdown(
    "These are kernel-side regression checks: they use fresh widgets and verify that LaTeX text, MathJSON, "
    "and stale-transport fallback all normalize to the expected canonical identifier."
))
identifier_regression

These are kernel-side regression checks: they use fresh widgets and verify that LaTeX text, MathJSON, and stale-transport fallback all normalize to the expected canonical identifier.

{'latex_text': 'a_1_2', 'mathjson': 'a_1_2', 'text_after_mathjson': 'x'}

## 2. `ExpressionInput` keeps known names atomic

Without a context, names like `velocity` could be parsed as a product of single-letter symbols. The visible widget below now starts empty on purpose: it is a real frontend demo, not a kernel-side preset.

That means the notebook keeps the browser-driven walkthrough separate from the later kernel-side regression cells. The visible widget is there for live exploration; the later assertions use fresh widgets so the notebook still executes cleanly in headless/kernel-only test runs.

In [23]:
expression_widget = ExpressionInput(
    context=ctx,
    aria_label="Expression demo input",
)
display(expression_widget)
display(Markdown(
    "This visible widget also starts empty. Type the example expression "+r'$\operatorname{Force}(\mathrm{theta\_x}) + 2\mathrm{velocity}$'+" manually, or run the optional browser-side "
    "injection cell below, then re-run the parse/inspection cell."
))
#BUG this mathlive feild has issues. Here are the issues:
# - It should not have "color" (disable by default)
# - It should not have "Evaluate" or "Simplify (disable by default)
# - It SHOULD list variables and functions from context in the drop-down menu
# Trying to write Force makes the recognizes "or" and transforms it into F \ve ce
# Mathlive should use "informed" information about the context and help with input!
# Design and implement this. 

#BUG This cell should be used as experimentation, not assertion
# A separate cell should use assertion (can reuse this field). 

This visible widget also starts empty. Type the example expression $\operatorname{Force}(\mathrm{theta\_x}) + 2\mathrm{velocity}$ manually, or run the optional browser-side injection cell below, then re-run the parse/inspection cell.

The next code cell is optional and only matters in a live notebook frontend. It injects the example expression into the visible widget so the following inspection cell can read the synced Python-side transport state.

In [17]:
display(Javascript(r"""
(() => {
  const selector = 'math-field[aria-label="Expression demo input"], textarea[aria-label="Expression demo input"]';
  const field = document.querySelector(selector);
  if (!field) {
    console.warn("Expression demo input not found. Run the display cell above first.");
    return;
  }
  const nextValue = String.raw`\operatorname{Force}(\mathrm{theta\_x}) + 2\mathrm{velocity}`;
  try {
    if (typeof field.setValue === "function") {
      field.setValue(nextValue, { silenceNotifications: false });
    } else {
      field.value = nextValue;
    }
  } catch (error) {
    field.value = nextValue;
  }
  field.dispatchEvent(new Event("input", { bubbles: true }));
  field.dispatchEvent(new Event("change", { bubbles: true }));
})();
"""))
display(Markdown(
    "In a live frontend, the browser just injected the example expression into the visible widget. "
    "Re-run the next cell to inspect the synced Python state."
))

<IPython.core.display.Javascript object>

In a live frontend, the browser just injected the example expression into the visible widget. Re-run the next cell to inspect the synced Python state.

In [24]:
expected_expression = Force(theta_x_literal) + 2 * velocity
expression_state = {
    "value": expression_widget.value,
    "math_json": expression_widget.math_json,
    "transport_valid": expression_widget.transport_valid,
    "transport_errors": expression_widget.transport_errors,
}

if expression_widget.value or expression_widget.math_json is not None:
    parsed_from_text = expression_widget.parse_value()
    sympy_latex = sp.latex(parsed_from_text, symbol_names=ctx.symbol_name_map(parsed_from_text))
    wrapper_latex = ctx.render_latex(parsed_from_text)

    assert parsed_from_text == expected_expression
    assert wrapper_latex == sympy_latex

    display(Markdown(f"Parsed SymPy expression: `{sp.sstr(parsed_from_text)}`"))
    display(Markdown(f"SymPy printer output: `${sympy_latex}$`"))
else:
    parsed_from_text = None
    sympy_latex = sp.latex(expected_expression, symbol_names=ctx.symbol_name_map(expected_expression))
    wrapper_latex = ctx.render_latex(expected_expression)
    display(Markdown(
        "The widget is still empty in this kernel-only execution. In a live frontend, type the example expression or "
        "run the optional browser-side injection cell above, then re-run this cell."
    ))
    display(Markdown(f"Reference LaTeX for the target expression: `${sympy_latex}$`"))

{
    "expected_expression": expected_expression,
    "parsed_expression": parsed_from_text,
    "sympy_latex": sympy_latex,
    "wrapper_latex": wrapper_latex,
    "visible_widget_state": expression_state,
}

AssertionError: 

## 3. MathJSON integration check

When the MathLive Compute Engine is available, the widget can send a structured MathJSON payload. The backend should parse that payload through the same `ExpressionContext` and recover the same SymPy expression as the visible example above.

SymPy is still the printer of record for the final rendered expression, and the visible walkthrough above only trusts structured transport when it matches the current visible field state.

This section intentionally uses a fresh widget so it tests transport parsing directly instead of mutating the earlier visible demo.

In [ ]:
expression_mathjson_widget = ExpressionInput(context=ctx)
expression_mathjson_widget.math_json = [
    "Add",
    ["Force", "theta__x"],
    ["Multiply", "velocity", 2],
]

parsed_from_mathjson = expression_mathjson_widget.parse_value()
assert parsed_from_mathjson == expected_expression

parsed_from_mathjson

## 4. `transport_manifest()` is a derived frontend snapshot

At this point the user-facing workflow is already working, so now the low-level transport is easier to place architecturally.

`ctx.transport_manifest(...)` is the JSON-safe payload sent to the frontend. It is **not** the primary Python authoring interface. The small schema worth knowing is:

- top level: `version`, `fieldRole`, `symbols`, `functions`
- symbol entries: `name`, `latex`, plus trigger metadata
- function entries: `name`, `latexHead`, `template`, and optional `arity`

That is enough for debugging or for understanding the browser contract without teaching the raw manifest before the conceptual model.


In [ ]:
manifest = ctx.transport_manifest(field_role="expression")
symbols_by_name = {entry["name"]: entry for entry in manifest["symbols"]}
functions_by_name = {entry["name"]: entry for entry in manifest["functions"]}

assert manifest["fieldRole"] == "expression"
assert symbols_by_name["velocity"]["latex"] == r"\mathrm{velocity}"
assert functions_by_name["Force"]["latexHead"] == r"\operatorname{Force}"
assert functions_by_name["Force"]["template"] == r"\operatorname{Force}(#0)"

{
    "fieldRole": manifest["fieldRole"],
    "velocity_entry": symbols_by_name["velocity"],
    "Force_entry": functions_by_name["Force"],
}


## 5. Some errors are intentional safeguards

A transport spelling such as `theta_x` is ambiguous when no context says whether it should mean a real subscript or a literal underscore inside the atom. In that situation the toolkit **raises** instead of guessing.

This is part of the overall design: fail early at the semantic boundary, not later inside unrelated plotting code.


In [ ]:
try:
    mathjson_to_identifier("theta_x")
except MathJSONParseError as exc:
    ambiguous_message = str(exc)
else:
    raise AssertionError("Expected an ambiguity error for an unregistered single-underscore name.")

assert "ambiguous" in ambiguous_message.lower()
assert mathjson_to_identifier("theta__x", context=ctx) == "theta__x"

ambiguous_message


## 6. Empty MathJSON means “no input”

The frontend may represent an empty field with a sentinel MathJSON payload such as `Nothing`. The backend should treat that as missing input, not as the number `0` and not as an identifier named `Nothing`.

In [ ]:
empty_expression_widget = ExpressionInput()
empty_expression_widget.math_json = "Nothing"

empty_identifier_widget = IdentifierInput()
empty_identifier_widget.math_json = "Nothing"

for widget, role in ((empty_expression_widget, "expression"), (empty_identifier_widget, "identifier")):
    try:
        widget.parse_value()
    except ValueError as exc:
        assert "required" in str(exc)
    else:
        raise AssertionError(f"Expected {role} parsing to reject empty/sentinel MathJSON.")

display(Markdown("Empty/sentinel MathJSON now behaves like missing input instead of parsing as `0` or `Nothing`."))

## Try your own variations

Good experiments after reading this notebook:

- register a new symbol such as `sigma_1` or `sigma__x` and compare the rendered LaTeX
- add a new function name such as `Energy_t` and inspect the transport manifest again
- change the widget text to plain canonical input (`Force(theta__x) + 2*velocity`) and compare it with the explicit LaTeX form
- compare what happens with `theta_x` before and after registering it in the context
